# Notebook 3 — Régression
**Variable cible : cholesterol** (mg/dL)


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold, cross_validate, RandomizedSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../health_lifestyle_dataset.csv')
df['gender_enc'] = (df['gender'] == 'Male').astype(int)
df['hypertension'] = ((df['systolic_bp'] >= 140) | (df['diastolic_bp'] >= 90)).astype(int)
df['bmi_cat'] = pd.cut(df['bmi'], bins=[0,18.5,25,30,100], labels=[0,1,2,3]).astype(int)
df['lifestyle_score'] = (
    df['daily_steps']/df['daily_steps'].max() + df['sleep_hours']/df['sleep_hours'].max() +
    df['water_intake_l']/df['water_intake_l'].max() - df['smoker'] - df['alcohol'] -
    df['bmi']/df['bmi'].max() - df['calories_consumed']/df['calories_consumed'].max())

# NOTE : hypercholesterol EXCLUE (data leakage)
features = ['age','bmi','daily_steps','sleep_hours','water_intake_l','calories_consumed',
            'smoker','alcohol','resting_hr','systolic_bp','diastolic_bp',
            'family_history','gender_enc','bmi_cat','hypertension','lifestyle_score']
X = df[features]; y = df['cholesterol']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train); X_te_sc = scaler.transform(X_test)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (80000, 16), Test: (20000, 16)


## 1. Modèles baseline (paramètres par défaut)

In [2]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
results = {}
for name, model, scaled in [
    ('LinearRegression', LinearRegression(), True),
    ('Ridge', Ridge(), True),
    ('Lasso', Lasso(), True),
    ('ElasticNet', ElasticNet(), True),
    ('RandomForest', RandomForestRegressor(max_depth=15, max_samples=0.5, n_jobs=-1, random_state=42), False),
    ('XGBoost', XGBRegressor(n_jobs=-1, random_state=42, verbosity=0), False)]:
    Xtr = X_tr_sc if scaled else X_train.values
    Xte = X_te_sc if scaled else X_test.values
    model.fit(Xtr, y_train)
    p = model.predict(Xte)
    r2 = r2_score(y_test, p)
    rmse = np.sqrt(mean_squared_error(y_test, p))
    mae = mean_absolute_error(y_test, p)
    results[name] = {'R2': r2, 'RMSE': rmse, 'MAE': mae}
    print(f'{name:20s} | R2={r2:.4f} | RMSE={rmse:.3f} | MAE={mae:.3f}')
pd.DataFrame(results).T.round(4)

LinearRegression     | R2=0.0001 | RMSE=43.328 | MAE=37.594
Ridge                | R2=0.0001 | RMSE=43.328 | MAE=37.594
Lasso                | R2=-0.0000 | RMSE=43.331 | MAE=37.596
ElasticNet           | R2=-0.0000 | RMSE=43.331 | MAE=37.596
RandomForest         | R2=-0.0050 | RMSE=43.439 | MAE=37.667
XGBoost              | R2=-0.0410 | RMSE=44.211 | MAE=38.134


,R2,RMSE,MAE
LinearRegression,0.0001,43.3282,37.5939
Ridge,0.0001,43.3282,37.5939
Lasso,-0.0000,43.3315,37.5958
ElasticNet,-0.0000,43.3315,37.5958
RandomForest,-0.0050,43.4386,37.6666
XGBoost,-0.0410,44.2114,38.1338


## 2. Tuning des hyperparamètres (RandomizedSearchCV)

In [3]:
# Ridge
rs = RandomizedSearchCV(Ridge(), {'alpha': np.logspace(-3,4,50)}, n_iter=20, cv=cv, scoring='r2', n_jobs=-1, random_state=42)
rs.fit(X_tr_sc, y_train)
print(f'Ridge tuné | alpha={rs.best_params_["alpha"]:.3f} | R2={r2_score(y_test, rs.best_estimator_.predict(X_te_sc)):.4f}')

# Lasso
ls = RandomizedSearchCV(Lasso(), {'alpha': np.logspace(-3,2,50)}, n_iter=20, cv=cv, scoring='r2', n_jobs=-1, random_state=42)
ls.fit(X_tr_sc, y_train)
best_lasso = ls.best_estimator_
print(f'Lasso tuné | alpha={ls.best_params_["alpha"]:.4f} | coefs nuls: {(best_lasso.coef_==0).sum()}/16')

# XGBoost
xgb_p = {'n_estimators':[100,200],'max_depth':[3,5,6],'learning_rate':[0.01,0.05,0.1],'subsample':[0.7,0.8,1.0]}
xs = RandomizedSearchCV(XGBRegressor(n_jobs=-1, random_state=42, verbosity=0), xgb_p, n_iter=20, cv=cv, scoring='r2', n_jobs=-1, random_state=42)
xs.fit(X_train.values, y_train)
print(f'XGBoost tuné | R2={r2_score(y_test, xs.best_estimator_.predict(X_test.values)):.4f}')

Ridge tuné | alpha=7196.857 | R2=0.0001
Lasso tuné | alpha=9.5410 | coefs nuls: 16/16
XGBoost tuné | R2=0.0001


## 3. Analyse : data leakage avec hypercholesterol

In [4]:
# Démonstration du data leakage
df['hypercholesterol'] = (df['cholesterol'] > 200).astype(int)
features_leak = features + ['hypercholesterol']
Xl = df[features_leak]; yl = df['cholesterol']
Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(Xl, yl, test_size=0.2, random_state=42)
from sklearn.linear_model import Ridge as R
scl = StandardScaler()
m = R().fit(scl.fit_transform(Xl_tr), yl_tr)
print(f'Avec hypercholesterol (data leakage) : R2 = {r2_score(yl_te, m.predict(scl.transform(Xl_te))):.4f}')
print(f'Sans hypercholesterol (correct)      : R2 = 0.0001')
print('→ Différence due au data leakage !')

Avec hypercholesterol (data leakage) : R2 = 0.6755
Sans hypercholesterol (correct)      : R2 = 0.0001
→ Différence due au data leakage !
